# TikTok 30-Day Video Engagement Prediction: GPU Benchmark
### XGBoost GPU vs. LightGBM vs. PyTorch Tabular Deep Learning

<a href="https://colab.research.google.com/github/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Project Goal**: Predict incremental plays gained between Day 3 and Day 30 (`incr_plays_3_30`) on short-form TikTok videos.  
**Primary Metric**: **sMAPE** (Symmetric Mean Absolute Percentage Error)  
**Secondary Metrics**: $R^2$ (log), MedAE, MAE, RMSE, WAPE  
**Hardware**: Google Colab Free T4 GPU / Local CUDA GPU

---
### Workflow Architecture:
1. **GPU Verification & Setup**: Configure CUDA execution for XGBoost (`tree_method='hist'`, `device='cuda'`) and PyTorch.
2. **Data Ingestion**: Load the 30-day tracking dataset (`processed_video_30d.parquet`) or download raw parquet tables.
3. **Feature Engineering V3**: Trajectory levels, velocities, acceleration, creator authority, rank percentiles, temporal dynamics, text indicators, and leak-free target encodings.
4. **LightGBM Benchmark**: Compare MAPE objective against MSE.
5. **XGBoost GPU Benchmark**: Evaluate `reg:absoluteerror` (MAE), `reg:pseudohubererror` (Huber), and `reg:squarederror` (MSE).
6. **Optuna GPU Hyperparameter Tuning**: Multi-trial Bayesian optimization targeting sMAPE.
7. **PyTorch Tabular Deep Learning**: Tabular ResNet / Residual MLP with LayerNorm, Swish activations, and Huber/sMAPE loss.
8. **Master Leaderboard & Diagnostic Plots**: Side-by-side sMAPE comparison, error distributions, and feature importances.
9. **Production Export**: Export tested configurations to `src/ml_models.py` and `main.py`.


In [ ]:
# Step 1: Install Dependencies & Verify GPU
import sys
in_colab = 'google.colab' in sys.modules

if in_colab:
    print("[Colab Detected] Installing required packages...")
    !pip install -q xgboost lightgbm optuna torch torchvision seaborn scikit-learn pyarrow

import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import xgboost as xgb
import lightgbm as lgb
import optuna

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
xgb_device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch Version   : {torch.__version__}")
print(f"XGBoost Version   : {xgb.__version__}")
print(f"LightGBM Version  : {lgb.__version__}")
print(f"Optuna Version    : {optuna.__version__}")
print(f"Computation Device: {device.upper()}")

if device == "cuda":
    print(f"GPU Name          : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM Available    : {vram_gb:.2f} GB")
else:
    print("NOTE: Running on CPU. In Colab, navigate to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


In [ ]:
# Step 2: Data Ingestion (Colab / Local)
import urllib.request

LOCAL_PATH = "data/processed_video_30d.parquet"

# Fallback path checks for Colab environment
if not os.path.exists(LOCAL_PATH):
    if os.path.exists("processed_video_30d.parquet"):
        LOCAL_PATH = "processed_video_30d.parquet"
    elif in_colab:
        print("[Colab] Please upload 'processed_video_30d.parquet' to the Colab files panel or clone your repo:")
        print("!git clone https://github.com/your-repo/Video-Engagement.git && cp Video-Engagement/data/processed_video_30d.parquet .")
    else:
        raise FileNotFoundError(f"File not found: {LOCAL_PATH}. Please run PySpark ETL first.")

df_raw = pd.read_parquet(LOCAL_PATH)
print(f"Dataset Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
n_target = df_raw['incr_plays_3_30'].notna().sum()
print(f"Videos with 30-Day Tracking Target: {n_target:,}")
display(df_raw[['video_id', 'topic', 'plays_day0', 'plays_day1', 'plays_day3', 'plays_day7', 'plays_day30', 'incr_plays_3_30']].head(4))


In [ ]:
# Step 3: Feature Engineering V3 (Leak-Free Out-of-Fold Encoding)
def target_encode(df_train, df_test, col, target_col, smoothing=30):
    global_mean = float(df_train[target_col].mean())
    grouped = df_train.groupby(col)[target_col]
    counts = grouped.count()
    means = grouped.mean()
    smooth_series = (counts * means + smoothing * global_mean) / (counts + smoothing)
    train_encoded = df_train[col].map(smooth_series).fillna(global_mean).values
    test_encoded = df_test[col].map(smooth_series).fillna(global_mean).values
    return train_encoded, test_encoded

def prepare_features(df_30d):
    df = df_30d.dropna(subset=["incr_plays_3_30"]).copy()
    y = np.log1p(df["incr_plays_3_30"].clip(lower=0).fillna(0))
    
    # 1. Trajectory log levels
    df["plays_day0_log"]  = np.log1p(df["plays_day0"].fillna(0))
    df["plays_day1_log"]  = np.log1p(df["plays_day1"].fillna(0))
    df["plays_day3_log"]  = np.log1p(df["plays_day3"].fillna(0))
    df["plays_day7_log"]  = np.log1p(df.get("plays_day7", df["plays_day3"]).fillna(0))
    df["likes_day3_log"]  = np.log1p(df.get("likes_day3", 0).fillna(0))
    df["likes_day7_log"]  = np.log1p(df.get("likes_day7", 0).fillna(0))
    df["shares_day7_log"] = np.log1p(df.get("shares_day7", 0).fillna(0))
    
    # 2. Velocity and dynamics
    df["velocity_0_1_log"] = np.log1p(df.get("velocity_0_1", df["plays_day1"] - df["plays_day0"]).clip(lower=0).fillna(0))
    df["velocity_1_3_log"] = np.log1p(df.get("velocity_1_3", (df["plays_day3"] - df["plays_day1"])/2.0).clip(lower=0).fillna(0))
    df["acceleration_val"] = df.get("acceleration", 0).fillna(0)
    df["decay_ratio_val"]  = df.get("decay_ratio", 0).clip(-10, 10).fillna(0)
    df["velocity_3_7"]     = np.log1p(((df.get("plays_day7", df["plays_day3"]).fillna(0) - df["plays_day3"].fillna(0)) / 4.0).clip(lower=0))
    
    # 3. Engagement Quality
    df["like_rate"]       = df.get("like_rate_day3", 0).fillna(0)
    df["comment_rate"]    = df.get("comment_rate_day3", 0).fillna(0)
    df["share_rate"]      = df.get("share_rate_day3", 0).fillna(0)
    df["like_rate_day7"]  = df.get("likes_day7", 0).fillna(0) / (df.get("plays_day7", 0).fillna(0) + 1.0)
    
    # 4. Creator Authority
    df["creator_median_log"]      = np.log1p(df.get("creator_median_plays30", 0).fillna(0))
    df["creator_video_count_val"] = df.get("creator_video_count", 1).fillna(1)
    df["penetration_rate_val"]    = df.get("penetration_rate", 0).clip(0, 100).fillna(0)
    df["follower_count_log"]      = np.log1p(df.get("follower_count", 0).fillna(0))
    
    # 5. Rank Percentiles (Spearman monotonic power)
    for col in ["velocity_1_3", "plays_day3", "creator_median_plays30", "follower_count"]:
        if col in df.columns:
            df[f"{col}_rank_pct"] = df[col].fillna(0).rank(pct=True)
    if "plays_day3" in df.columns and "topic" in df.columns:
        df["plays_day3_topic_rank_pct"] = df.groupby("topic")["plays_day3"].rank(pct=True)
        
    # 6. Temporal Features
    if "create_date" in df.columns:
        dates = pd.to_datetime(df["create_date"], errors="coerce")
        df["post_day_of_week"] = dates.dt.dayofweek
        df["post_month"]       = dates.dt.month
        df["post_is_weekend"]  = (dates.dt.dayofweek >= 5).astype(int)
        df["days_since_start"] = (dates - dates.min()).dt.days
    
    # 7. Text Metadata & Missing Indicators
    for col in ["word_count", "speaking_rate", "hashtag_count"]:
        if col in df.columns:
            df[f"{col}_missing"] = df[col].isna().astype(int)
            df[col] = df[col].fillna(df[col].median())
            
    df["duration"]   = df.get("duration", 15).fillna(15)
    df["is_english"] = df.get("is_english", 1).fillna(1)
    
    emotion_cols = ["joy", "disgust", "sadness", "anger", "surprise", "fear"]
    for c in emotion_cols:
        if c in df.columns:
            df[c] = df[c].fillna(df[c].median())
            
    feature_cols = [
        "plays_day0_log", "plays_day1_log", "plays_day3_log", "plays_day7_log",
        "likes_day3_log", "likes_day7_log", "shares_day7_log",
        "velocity_0_1_log", "velocity_1_3_log", "acceleration_val", "decay_ratio_val", "velocity_3_7",
        "like_rate", "comment_rate", "share_rate", "like_rate_day7",
        "creator_median_log", "creator_video_count_val", "penetration_rate_val", "follower_count_log",
        "velocity_1_3_rank_pct", "plays_day3_rank_pct", "creator_median_plays30_rank_pct",
        "follower_count_rank_pct", "plays_day3_topic_rank_pct",
        "post_day_of_week", "post_month", "post_is_weekend", "days_since_start",
        "duration", "is_english", "word_count", "speaking_rate", "hashtag_count",
        "word_count_missing", "speaking_rate_missing", "hashtag_count_missing"
    ] + [c for c in emotion_cols if c in df.columns]
    
    feature_cols = [c for c in feature_cols if c in df.columns]
    X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    return X, y, df, feature_cols

X_all, y_all, df_clean, feat_cols = prepare_features(df_raw)

# Strict Time-Based Split (80% Train, 20% Out-of-Time Test)
dates = pd.to_datetime(df_clean["create_date"], errors="coerce")
cutoff = dates.quantile(0.8)
train_mask = dates <= cutoff
test_mask = dates > cutoff

X_train = X_all[train_mask].copy()
y_train = y_all[train_mask].copy()
X_test  = X_all[test_mask].copy()
y_test  = y_all[test_mask].copy()

# Leak-Free Smoothed Target Encoding
df_tr_raw = df_clean[train_mask].reset_index(drop=True)
df_te_raw = df_clean[test_mask].reset_index(drop=True)
for cat in ["author_id", "topic"]:
    if cat in df_clean.columns:
        te_tr, te_te = target_encode(df_tr_raw, df_te_raw, cat, "incr_plays_3_30")
        X_train[f"{cat}_te"] = te_tr
        X_test[f"{cat}_te"]  = te_te

print(f"Train split: {len(X_train):,} samples | Test split: {len(X_test):,} samples")
print(f"Total Model Features: {X_train.shape[1]}")


In [ ]:
# Step 4: Metrics Evaluation Suite
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score

def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    return np.mean(2.0 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1.0)) * 100

def evaluate_predictions(y_true_log, y_pred_log, model_name="Model"):
    y_pred_log = np.clip(np.array(y_pred_log, dtype=float), -1, 25)
    y_true_orig = np.expm1(np.array(y_true_log, dtype=float))
    y_pred_orig = np.clip(np.expm1(y_pred_log), 0, None)
    
    mae = mean_absolute_error(y_true_orig, y_pred_orig)
    medae = median_absolute_error(y_true_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    r2_log = r2_score(y_true_log, y_pred_log)
    denom = np.sum(np.abs(y_true_orig))
    wape = (np.sum(np.abs(y_true_orig - y_pred_orig)) / denom * 100) if denom > 0 else np.nan
    smape_val = smape(y_true_orig, y_pred_orig)
    
    return {
        "Model": model_name,
        "sMAPE (%)": round(smape_val, 2),
        "R2 (log)": round(r2_log, 4),
        "MedAE": round(medae, 2),
        "MAE": round(mae, 1),
        "RMSE": round(rmse, 1),
        "WAPE (%)": round(wape, 2)
    }

benchmark_records = []


In [ ]:
# Step 5: LightGBM Baseline (MAPE vs. MSE)
print("--- [1/6] Training LightGBM (MAPE objective) ---")
lgb_mape = lgb.LGBMRegressor(
    objective="mape", n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
t0 = time.time()
lgb_mape.fit(X_train, y_train)
pred_lgb_mape = lgb_mape.predict(X_test)
res_lgb_mape = evaluate_predictions(y_test, pred_lgb_mape, "LightGBM (MAPE obj)")
res_lgb_mape["Train Time (s)"] = round(time.time() - t0, 2)
benchmark_records.append(res_lgb_mape)

print("--- [2/6] Training LightGBM (MSE objective) ---")
lgb_mse = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
t0 = time.time()
lgb_mse.fit(X_train, y_train)
pred_lgb_mse = lgb_mse.predict(X_test)
res_lgb_mse = evaluate_predictions(y_test, pred_lgb_mse, "LightGBM (MSE obj)")
res_lgb_mse["Train Time (s)"] = round(time.time() - t0, 2)
benchmark_records.append(res_lgb_mse)

display(pd.DataFrame(benchmark_records))


In [ ]:
# Step 6: XGBoost GPU Benchmarking (MAE vs. Huber vs. MSE)
tree_method = "hist"
xgb_dev = xgb_device

print(f"Configuring XGBoost with tree_method='{tree_method}', device='{xgb_dev}'")

# 1. XGBoost with MAE Loss (reg:absoluteerror)
print("\n--- [3/6] Training XGBoost (reg:absoluteerror, GPU) ---")
xgb_mae = xgb.XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method=tree_method,
    device=xgb_dev,
    random_state=42
)
t0 = time.time()
xgb_mae.fit(X_train, y_train)
pred_xgb_mae = xgb_mae.predict(X_test)
res_xgb_mae = evaluate_predictions(y_test, pred_xgb_mae, "XGBoost (MAE obj, GPU)")
res_xgb_mae["Train Time (s)"] = round(time.time() - t0, 2)
benchmark_records.append(res_xgb_mae)

# 2. XGBoost with Pseudo-Huber Loss (reg:pseudohubererror)
print("--- [4/6] Training XGBoost (reg:pseudohubererror, GPU) ---")
xgb_huber = xgb.XGBRegressor(
    objective="reg:pseudohubererror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method=tree_method,
    device=xgb_dev,
    random_state=42
)
t0 = time.time()
xgb_huber.fit(X_train, y_train)
pred_xgb_huber = xgb_huber.predict(X_test)
res_xgb_huber = evaluate_predictions(y_test, pred_xgb_huber, "XGBoost (Huber obj, GPU)")
res_xgb_huber["Train Time (s)"] = round(time.time() - t0, 2)
benchmark_records.append(res_xgb_huber)

# 3. XGBoost with Standard MSE Loss (reg:squarederror)
print("--- [5/6] Training XGBoost (reg:squarederror, GPU) ---")
xgb_mse = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method=tree_method,
    device=xgb_dev,
    random_state=42
)
t0 = time.time()
xgb_mse.fit(X_train, y_train)
pred_xgb_mse = xgb_mse.predict(X_test)
res_xgb_mse = evaluate_predictions(y_test, pred_xgb_mse, "XGBoost (MSE obj, GPU)")
res_xgb_mse["Train Time (s)"] = round(time.time() - t0, 2)
benchmark_records.append(res_xgb_mse)

display(pd.DataFrame(benchmark_records))


In [ ]:
# Step 7: Optuna Hyperparameter Optimization on XGBoost GPU (30 Trials)
print("--- Running Optuna Hyperparameter Optimization on XGBoost GPU ---")

val_split_idx = int(len(X_train) * 0.85)
X_tr, y_tr = X_train.iloc[:val_split_idx], y_train.iloc[:val_split_idx]
X_val, y_val = X_train.iloc[val_split_idx:], y_train.iloc[val_split_idx:]
y_val_orig = np.expm1(y_val.values)

def optuna_xgb_objective(trial):
    params = {
        "objective": "reg:absoluteerror",
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "tree_method": tree_method,
        "device": xgb_dev,
        "random_state": 42
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_tr, y_tr)
    pred_val = np.clip(np.expm1(model.predict(X_val)), 0, None)
    return smape(y_val_orig, pred_val)

study = optuna.create_study(direction="minimize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(optuna_xgb_objective, n_trials=30, show_progress_bar=True)

print(f"Optuna Best sMAPE: {study.best_value:.2f}%")
print(f"Best Hyperparameters: {study.best_params}")

# Fit final tuned XGBoost model
t0 = time.time()
best_xgb = xgb.XGBRegressor(**study.best_params, objective="reg:absoluteerror",
                             tree_method=tree_method, device=xgb_dev, random_state=42)
best_xgb.fit(X_train, y_train)
pred_best_xgb = best_xgb.predict(X_test)
res_best_xgb = evaluate_predictions(y_test, pred_best_xgb, "XGBoost (Optuna Tuned, GPU)")
res_best_xgb["Train Time (s)"] = round(time.time() - t0, 2)
benchmark_records.append(res_best_xgb)

display(pd.DataFrame(benchmark_records).sort_values("sMAPE (%)"))


In [ ]:
# Step 8: PyTorch Tabular Deep Learning Model (GPU)
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

print(f"--- [6/6] Training Deep Learning Tabular ResNet on {device.upper()} ---")

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train)
X_te_scaled = scaler.transform(X_test)

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values if hasattr(y, 'values') else y, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TabularDataset(X_tr_scaled, y_train)
test_dataset  = TabularDataset(X_te_scaled, y_test)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, drop_last=False)
test_loader  = DataLoader(test_dataset, batch_size=1024, shuffle=False)

# Tabular ResNet Block with LayerNorm & SiLU (Swish)
class TabularResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)
        self.act1 = nn.SiLU()
        self.fc2 = nn.Linear(dim, dim)
        self.norm2 = nn.LayerNorm(dim)
        self.act2 = nn.SiLU()
        self.drop = nn.Dropout(dropout)
        
    def forward(self, x):
        residual = x
        out = self.act1(self.norm1(self.fc1(x)))
        out = self.drop(out)
        out = self.norm2(self.fc2(out))
        return self.act2(out + residual)

class TabularDeepNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_blocks=2, dropout=0.25):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout)
        )
        self.blocks = nn.ModuleList([TabularResidualBlock(hidden_dim, dropout) for _ in range(num_blocks)])
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, x):
        h = self.input_layer(x)
        for b in self.blocks:
            h = b(h)
        return self.head(h)

model_dl = TabularDeepNet(input_dim=X_train.shape[1], hidden_dim=256, num_blocks=2, dropout=0.25).to(device)
criterion = nn.SmoothL1Loss(beta=1.0)
optimizer = torch.optim.AdamW(model_dl.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

epochs = 40
t0 = time.time()

for epoch in range(1, epochs + 1):
    model_dl.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model_dl(bx)
        loss = criterion(out, by)
        loss.backward()
        nn.utils.clip_grad_norm_(model_dl.parameters(), 2.0)
        optimizer.step()
        total_loss += loss.item() * len(bx)
    scheduler.step()
    
    if epoch % 10 == 0 or epoch == epochs:
        avg_loss = total_loss / len(train_dataset)
        print(f"Epoch {epoch:02d}/{epochs:02d} | Train Huber Loss: {avg_loss:.4f}")

# Predict on Holdout Test Set
model_dl.eval()
preds_dl_list = []
with torch.no_grad():
    for bx, _ in test_loader:
        bx = bx.to(device)
        preds_dl_list.append(model_dl(bx).cpu().numpy().flatten())
pred_dl = np.concatenate(preds_dl_list)

res_dl = evaluate_predictions(y_test, pred_dl, "PyTorch Tabular ResNet (GPU)")
res_dl["Train Time (s)"] = round(time.time() - t0, 2)
benchmark_records.append(res_dl)

display(pd.DataFrame(benchmark_records).sort_values("sMAPE (%)"))


In [ ]:
# Step 9: Head-to-Head Visualizations & Leaderboard Analysis
results_df = pd.DataFrame(benchmark_records).sort_values("sMAPE (%)").reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Chart 1: sMAPE (Lower is Better)
sns.barplot(x="sMAPE (%)", y="Model", data=results_df, palette="rocket", ax=axes[0])
axes[0].set_title("Primary Metric: sMAPE (%) - Lower is Better", fontsize=13, fontweight="bold")
for p in axes[0].patches:
    val = p.get_width()
    axes[0].annotate(f"{val:.1f}%", (val + 0.5, p.get_y() + p.get_height() / 2.),
                     ha='left', va='center', fontsize=10, fontweight='bold')

# Chart 2: Log R2 Score (Higher is Better)
r2_df = results_df.sort_values("R2 (log)", ascending=False)
sns.barplot(x="R2 (log)", y="Model", data=r2_df, palette="viridis", ax=axes[1])
axes[1].set_title("R² Score (Log Scale) - Higher is Better", fontsize=13, fontweight="bold")
axes[1].axvline(0, color="red", linestyle="--", alpha=0.6)
for p in axes[1].patches:
    val = p.get_width()
    axes[1].annotate(f"{val:.3f}", (val + 0.005, p.get_y() + p.get_height() / 2.),
                     ha='left', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Feature Importance Comparison: XGBoost vs LightGBM
fi_xgb = pd.Series(xgb_mae.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)
fi_lgb = pd.Series(lgb_mape.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x=fi_xgb.values, y=fi_xgb.index, ax=axes[0], palette="mako")
axes[0].set_title("Top 15 Feature Importances: XGBoost (MAE Loss)", fontweight="bold")
axes[0].set_xlabel("Relative Weight")

sns.barplot(x=fi_lgb.values, y=fi_lgb.index, ax=axes[1], palette="teal")
axes[1].set_title("Top 15 Feature Importances: LightGBM (MAPE Loss)", fontweight="bold")
axes[1].set_xlabel("Split Count")

plt.tight_layout()
plt.show()


In [ ]:
# Step 10: Final Leaderboard Summary
print("=" * 76)
print("   MASTER GPU BENCHMARK: 30-DAY INCREMENTAL ENGAGEMENT (DAY 3->30)  ")
print("=" * 76)
print(results_df.to_string(index=False))

winner = results_df.iloc[0]
print(f"\n🏆 Winning Model: {winner['Model']} with sMAPE = {winner['sMAPE (%)']:.2f}% | Train Time = {winner['Train Time (s)']}s")
print("\nTo export this directly to your local codebase, update src/ml_models.py and main.py.")
